In [1]:
# read parket
import pyarrow.parquet as pq
import pandas as pd
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy import stats

plt.style.use("seaborn-v0_8")

In [62]:
import unicodedata
import requests

# Data

## Sources

**Datos principales incasol**

https://habitatge.gencat.cat/ca/dades/indicadors_estadistiques/estadistiques_de_construccio_i_mercat_immobiliari/mercat_de_lloguer/lloguers-barcelona-per-districtes-i-barris/index.html?utm_source=chatgpt.com#googtrans(ca|es)

**IPC - Indice de Precio del Consumidor**

https://www.ine.es/jaxiT3/Datos.htm?t=24078

**Euribor**

https://app.bde.es/bie_www/bie_wwwias/xml/Arranque.html?Idioma=es

**Indice socioeconomico territorial - Barrios**
https://www.idescat.cat/dades/obertes/ist?lang=es&utm_source=chatgpt.com

In [2]:
# read parquet file

districts_raw = pd.read_parquet('district_panel_2000_2025.parquet')
districts_raw.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   district_code  1040 non-null   int64  
 1   district       1040 non-null   object 
 2   year           1040 non-null   int64  
 3   quarter        1040 non-null   int64  
 4   num_contracts  1030 non-null   Int64  
 5   avg_rent       1030 non-null   float64
 6   avg_rent_m2    1030 non-null   float64
 7   avg_surface    1030 non-null   float64
dtypes: Int64(1), float64(3), int64(3), object(1)
memory usage: 66.1+ KB


In [3]:
# read parquet file

neighborhood_raw = pd.read_parquet('rental_panel_barris_2014_2025.parquet')
neighborhood_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   neighborhood_code  3504 non-null   int64  
 1   neighborhood       3504 non-null   object 
 2   year               3504 non-null   int64  
 3   quarter            3504 non-null   int64  
 4   num_contracts      3429 non-null   Int64  
 5   avg_rent           3283 non-null   float64
 6   avg_rent_m2        3285 non-null   float64
 7   avg_surface        3400 non-null   float64
dtypes: Int64(1), float64(3), int64(3), object(1)
memory usage: 222.6+ KB


## Variable temporal

In [4]:
def create_time_index(df):
    df = df.copy()
    df["date"] = pd.PeriodIndex.from_fields(year=df["year"],quarter=df["quarter"]).to_timestamp()
    df = df.sort_values(["district_code" if "district_code" in df.columns else "neighborhood_code", "date"])
    return df

In [5]:
districts = create_time_index(districts_raw)
neighborhood = create_time_index(neighborhood_raw)

In [6]:
districts.head()

,district_code,district,year,quarter,num_contracts,avg_rent,avg_rent_m2,avg_surface,date
0,1,Ciutat Vella,2000,1,397,286.9246,4.998756,60.976331,2000-01-01
1,1,Ciutat Vella,2000,2,355,301.3140,5.401631,65.389937,2000-04-01
2,1,Ciutat Vella,2000,3,373,338.7222,5.958517,66.824390,2000-07-01
3,1,Ciutat Vella,2000,4,415,370.0641,5.758672,65.384211,2000-10-01
40,1,Ciutat Vella,2001,1,350,345.2979,6.459721,62.393035,2001-01-01


## Variables claves

In [7]:
def add_features(df, group_col):
    df = df.copy()

    df["contract_growth_yoy"] = df.groupby(group_col)["num_contracts"].pct_change(4, fill_method=None)
    df["rent_growth_yoy"] = df.groupby(group_col)["avg_rent"].pct_change(4, fill_method=None)
    df["rent_m2_growth_yoy"] = df.groupby(group_col)["avg_rent_m2"].pct_change(4, fill_method=None)

    df["contract_growth_qoq"] = df.groupby(group_col)["num_contracts"].pct_change(1, fill_method=None)

    df["rent_m2_lag1"] = df.groupby(group_col)["avg_rent_m2"].shift(1)
    df["rent_m2_lag4"] = df.groupby(group_col)["avg_rent_m2"].shift(4)

    df["post_regulation"] = (df["date"] >= "2020-10-01").astype(int)
    df["covid_dummy"] = ((df["date"] >= "2020-03-01") & (df["date"] <= "2021-06-01")).astype(int)

    return df



In [8]:
districts = add_features(districts, "district_code")
neighborhood = add_features(neighborhood, "neighborhood_code")

In [9]:
districts.tail()

,district_code,district,year,quarter,num_contracts,avg_rent,avg_rent_m2,avg_surface,date,contract_growth_yoy,rent_growth_yoy,rent_m2_growth_yoy,contract_growth_qoq,rent_m2_lag1,rent_m2_lag4,post_regulation,covid_dummy
999,10,Sant Martí,2024,4,810,1075.102593,15.755760,69.880637,2024-10-01,-0.154489,-0.043528,-0.057408,-0.00978,15.899334,16.715352,1,0
1036,10,Sant Martí,2025,1,997,1023.610040,16.014624,69.633293,2025-01-01,-0.052281,-0.102335,-0.046347,0.230864,15.755760,16.792924,1,0
1037,10,Sant Martí,2025,2,889,1098.616254,16.330055,68.788146,2025-04-01,-0.038919,0.015591,0.000165,-0.108325,16.014624,16.327357,1,0
1038,10,Sant Martí,2025,3,899,1120.271724,16.708644,68.573964,2025-07-01,0.099022,0.029472,0.050902,0.011249,16.330055,15.899334,1,0
1039,10,Sant Martí,2025,4,<NA>,NaN,NaN,NaN,2025-10-01,<NA>,NaN,NaN,<NA>,16.708644,15.755760,1,0


## Ajuste de precios nominales a precios reales
Dado que el análisis cubre un periodo largo, no resulta suficiente trabajar únicamente con precios nominales (avg_rent y avg_rent_m2), ya que su evolución recoge tanto cambios reales del mercado como el efecto acumulado de la inflación. Por ello, además de conservar las variables originales en euros corrientes, se incorporará una versión deflactada en euros constantes de un año base.

Este ajuste permite interpretar con mayor precisión la evolución del alquiler a lo largo del tiempo. En particular, aunque avg_rent_m2 ya controla por el tamaño medio de la vivienda, sigue siendo una medida nominal y, por tanto, también puede verse afectada por la inflación general. Trabajar con ambas versiones, nominal y real, aporta una visión más completa y rigurosa del comportamiento del mercado.

- usar IPC INE como serie principal,
- construir una versión real de avg_rent y avg_rent_m2,


#### EL IPC solo comienza en el 2002

Mantener el análisis nominal para todo el periodo 2000–2025.
Hacer el análisis en precios reales desde 2002 en adelante.
Explicar en el markdown que la serie real arranca en 2002 por el cambio de sistema/base del IPC.
Esta opción es muy sólida metodológicamente.

In [10]:
ipc = pd.read_csv(
    "ipc_ine.csv",
    sep=";",
    encoding="latin-1",
    decimal=","
)


In [11]:
print(ipc.head())
print(ipc.columns)
print(ipc.shape)

  Comunidades y Ciudades Autónomas Grupos COICOP 2011 Tipo de dato  Periodo  \
0                      09 Cataluña     Índice general       Índice  2026M02   
1                      09 Cataluña     Índice general       Índice  2026M01   
2                      09 Cataluña     Índice general       Índice  2025M12   
3                      09 Cataluña     Índice general       Índice  2025M11   
4                      09 Cataluña     Índice general       Índice  2025M10   

     Total  
0  101.108  
1  100.619  
2  101.019  
3  100.706  
4  100.453  
Index(['Comunidades y Ciudades Autónomas', 'Grupos COICOP 2011',
       'Tipo de dato', 'Periodo', 'Total'],
      dtype='object')
(314, 5)


In [12]:
ipc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 314 entries, 0 to 313
Data columns (total 5 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Comunidades y Ciudades Autónomas  314 non-null    object 
 1   Grupos COICOP 2011                314 non-null    object 
 2   Tipo de dato                      314 non-null    object 
 3   Periodo                           314 non-null    object 
 4   Total                             290 non-null    float64
dtypes: float64(1), object(4)
memory usage: 12.4+ KB


### Limpiamos el ipc para que solo muestre las columnas relevantes


In [13]:
ipc = ipc.rename(columns={
    "Comunidades y Ciudades Autónomas": "region",
    "Grupos COICOP 2011": "group",
    "Tipo de dato": "data_type",
    "Periodo": "period",
    "Total": "ipc_index"
})

# Crear year y month a partir de strings tipo 2026M02
ipc["year"] = ipc["period"].str[:4].astype(int)
ipc["month"] = ipc["period"].str[-2:].astype(int)

# Crear fecha mensual
ipc["date"] = pd.to_datetime(
    ipc["year"].astype(str) + "-" + ipc["month"].astype(str) + "-01"
)

# Ordenar cronológicamente
ipc = ipc.sort_values("date").reset_index(drop=True)

# Dejar solo columnas útiles
ipc = ipc[["date", "ipc_index", "year", "month"]]



In [14]:
print(ipc.head())
print(ipc.tail())
print(ipc.shape)

        date  ipc_index  year  month
0 2000-01-01        NaN  2000      1
1 2000-02-01        NaN  2000      2
2 2000-03-01        NaN  2000      3
3 2000-04-01        NaN  2000      4
4 2000-05-01        NaN  2000      5
          date  ipc_index  year  month
309 2025-10-01    100.453  2025     10
310 2025-11-01    100.706  2025     11
311 2025-12-01    101.019  2025     12
312 2026-01-01    100.619  2026      1
313 2026-02-01    101.108  2026      2
(314, 4)


### Convertir a trimestral

In [15]:
# Nos quedamos solo con meses con IPC válido
ipc_valid = ipc.dropna(subset=["ipc_index"]).copy()

# Crear quarter
ipc_valid["quarter"] = ipc_valid["date"].dt.quarter

# Agregar a trimestre usando la media de los 3 meses
ipc_quarterly = (
    ipc_valid
    .groupby(["year", "quarter"], as_index=False)
    .agg(ipc_index_q=("ipc_index", "mean"))
)

# Crear una fecha trimestral alineada con tus paneles
ipc_quarterly["date"] = pd.to_datetime(
    ipc_quarterly["year"].astype(str) + "-" +
    ((ipc_quarterly["quarter"] - 1) * 3 + 1).astype(str) + "-01"
)

# Reordenar columnas
ipc_quarterly = ipc_quarterly[["date", "year", "quarter", "ipc_index_q"]]



In [16]:
print(ipc_quarterly.head())
print(ipc_quarterly.tail())
print(ipc_quarterly.shape)
print(ipc_quarterly.info())

        date  year  quarter  ipc_index_q
0 2002-01-01  2002        1    56.727667
1 2002-04-01  2002        2    57.955667
2 2002-07-01  2002        3    57.872667
3 2002-10-01  2002        4    58.812000
4 2003-01-01  2003        1    59.107667
         date  year  quarter  ipc_index_q
92 2025-01-01  2025        1    98.987000
93 2025-04-01  2025        2    99.982667
94 2025-07-01  2025        3   100.304333
95 2025-10-01  2025        4   100.726000
96 2026-01-01  2026        1   100.863500
(97, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         97 non-null     datetime64[ns]
 1   year         97 non-null     int64         
 2   quarter      97 non-null     int32         
 3   ipc_index_q  97 non-null     float64       
dtypes: datetime64[ns](1), float64(1), int32(1), int64(1)
memory usage: 2.8 KB
None


## Merge ipc con district y neighborhood


In [17]:
# MERGE IPC trimestral
districts_ipc = districts.merge(
    ipc_quarterly[["date", "ipc_index_q"]],
    on="date",
    how="left"
)

neighborhood_ipc = neighborhood.merge(
    ipc_quarterly[["date", "ipc_index_q"]],
    on="date",
    how="left"
)



In [18]:
# Revisión rápida
print(districts_ipc[["date", "avg_rent", "avg_rent_m2", "ipc_index_q"]].head())
print(neighborhood_ipc[["date", "avg_rent", "avg_rent_m2", "ipc_index_q"]].head())

# Cuántos NaN quedaron en el IPC tras el merge
print("NaN IPC districts:", districts_ipc["ipc_index_q"].isna().sum())
print("NaN IPC neighborhood:", neighborhood_ipc["ipc_index_q"].isna().sum())

        date  avg_rent  avg_rent_m2  ipc_index_q
0 2000-01-01  286.9246     4.998756          NaN
1 2000-04-01  301.3140     5.401631          NaN
2 2000-07-01  338.7222     5.958517          NaN
3 2000-10-01  370.0641     5.758672          NaN
4 2001-01-01  345.2979     6.459721          NaN
        date    avg_rent  avg_rent_m2  ipc_index_q
0 2014-01-01  589.548624    10.760000    78.679000
1 2014-04-01  550.631711    10.520000    79.547000
2 2014-07-01  576.454232     9.840000    79.032333
3 2014-10-01  596.995494    10.810000    79.193333
4 2015-01-01  601.307029    10.598882    78.153667
NaN IPC districts: 80
NaN IPC neighborhood: 0


In [19]:
print(districts_ipc.info())
print(neighborhood_ipc.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   district_code        1040 non-null   int64         
 1   district             1040 non-null   object        
 2   year                 1040 non-null   int64         
 3   quarter              1040 non-null   int64         
 4   num_contracts        1030 non-null   Int64         
 5   avg_rent             1030 non-null   float64       
 6   avg_rent_m2          1030 non-null   float64       
 7   avg_surface          1030 non-null   float64       
 8   date                 1040 non-null   datetime64[ns]
 9   contract_growth_yoy  990 non-null    Float64       
 10  rent_growth_yoy      990 non-null    float64       
 11  rent_m2_growth_yoy   990 non-null    float64       
 12  contract_growth_qoq  1020 non-null   Float64       
 13  rent_m2_lag1         1030 non-nul

## Agregar avg_rent_real y avg_rent_m2_real

In [20]:
def add_real_rent_variables(df, rent_col="avg_rent", rent_m2_col="avg_rent_m2", ipc_col="ipc_index_q"):
    """
    Añade variables de precio real a un dataframe usando IPC trimestral.

    Fórmula:
        precio_real = precio_nominal / (IPC / 100)

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame que contiene las columnas de alquiler e IPC.
    rent_col : str
        Nombre de la columna de alquiler medio nominal.
    rent_m2_col : str
        Nombre de la columna de alquiler medio nominal por m².
    ipc_col : str
        Nombre de la columna del IPC trimestral.

    Returns
    -------
    pd.DataFrame
        Copia del dataframe con:
        - avg_rent_real
        - avg_rent_m2_real
    """
    out = df.copy()

    out["avg_rent_real_2025base"] = out[rent_col] / (out[ipc_col] / 100)
    out["avg_rent_m2_real_2025base"] = out[rent_m2_col] / (out[ipc_col] / 100)

    return out




In [21]:
# Aplicar a ambos dataframes
districts_ipc = add_real_rent_variables(districts_ipc)
neighborhood_ipc = add_real_rent_variables(neighborhood_ipc)

# Revisión rápida
print(districts_ipc[["date", "avg_rent", "avg_rent_real_2025base", "avg_rent_m2", "avg_rent_m2_real_2025base"]].head(10))
print(neighborhood_ipc[["date", "avg_rent", "avg_rent_real_2025base", "avg_rent_m2", "avg_rent_m2_real_2025base"]].head(10))

        date  avg_rent  avg_rent_real_2025base  avg_rent_m2  \
0 2000-01-01  286.9246                     NaN     4.998756   
1 2000-04-01  301.3140                     NaN     5.401631   
2 2000-07-01  338.7222                     NaN     5.958517   
3 2000-10-01  370.0641                     NaN     5.758672   
4 2001-01-01  345.2979                     NaN     6.459721   
5 2001-04-01  352.7870                     NaN     6.087407   
6 2001-07-01  342.2632                     NaN     6.448391   
7 2001-10-01  360.3416                     NaN     6.659629   
8 2002-01-01  433.4590              764.105110     7.533465   
9 2002-04-01  398.9955              688.449505     7.014424   

   avg_rent_m2_real_2025base  
0                        NaN  
1                        NaN  
2                        NaN  
3                        NaN  
4                        NaN  
5                        NaN  
6                        NaN  
7                        NaN  
8                  13.28005

In [22]:
print(districts_ipc.info())
print(neighborhood_ipc.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   district_code              1040 non-null   int64         
 1   district                   1040 non-null   object        
 2   year                       1040 non-null   int64         
 3   quarter                    1040 non-null   int64         
 4   num_contracts              1030 non-null   Int64         
 5   avg_rent                   1030 non-null   float64       
 6   avg_rent_m2                1030 non-null   float64       
 7   avg_surface                1030 non-null   float64       
 8   date                       1040 non-null   datetime64[ns]
 9   contract_growth_yoy        990 non-null    Float64       
 10  rent_growth_yoy            990 non-null    float64       
 11  rent_m2_growth_yoy         990 non-null    float64       
 12  contra

In [23]:
# Verificar duplicados en districts
district_dups = districts_ipc.duplicated(subset=["district_code", "date"]).sum()
print("Duplicados en districts:", district_dups)

# Verificar duplicados en neighborhood
neighborhood_dups = neighborhood_ipc.duplicated(subset=["neighborhood_code", "date"]).sum()
print("Duplicados en neighborhood:", neighborhood_dups)




Duplicados en districts: 0
Duplicados en neighborhood: 0


## Añadir real yoy

In [24]:
def add_real_yoy_growth(
    df,
    group_col,
    rent_real_col="avg_rent_real_2025base",
    rent_m2_real_col="avg_rent_m2_real_2025base"
):
    """
    Añade growth rates YoY reales a un panel trimestral.

    Fórmula:
        growth_yoy = (x_t / x_t-4) - 1

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame panel.
    group_col : str
        Columna identificadora de unidad territorial.
    rent_real_col : str
        Columna de alquiler real.
    rent_m2_real_col : str
        Columna de alquiler real por m².

    Returns
    -------
    pd.DataFrame
        Copia del dataframe con:
        - rent_real_growth_yoy
        - rent_m2_real_growth_yoy
    """
    out = df.copy()
    out = out.sort_values([group_col, "date"]).copy()

    out["rent_real_growth_yoy"] = (
        out.groupby(group_col)[rent_real_col]
        .pct_change(periods=4, fill_method=None)
    )

    out["rent_m2_real_growth_yoy"] = (
        out.groupby(group_col)[rent_m2_real_col]
        .pct_change(periods=4, fill_method=None)
    )

    return out

In [25]:
districts_ipc = add_real_yoy_growth(
    districts_ipc,
    group_col="district_code"
)

neighborhood_ipc = add_real_yoy_growth(
    neighborhood_ipc,
    group_col="neighborhood_code"
)

In [26]:
print(districts_ipc[["rent_real_growth_yoy", "rent_m2_real_growth_yoy"]].notna().sum())
print(neighborhood_ipc[["rent_real_growth_yoy", "rent_m2_real_growth_yoy"]].notna().sum())



rent_real_growth_yoy       910
rent_m2_real_growth_yoy    910
dtype: int64
rent_real_growth_yoy       2916
rent_m2_real_growth_yoy    2913
dtype: int64


In [27]:
print(districts_ipc.info())
print(neighborhood_ipc.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   district_code              1040 non-null   int64         
 1   district                   1040 non-null   object        
 2   year                       1040 non-null   int64         
 3   quarter                    1040 non-null   int64         
 4   num_contracts              1030 non-null   Int64         
 5   avg_rent                   1030 non-null   float64       
 6   avg_rent_m2                1030 non-null   float64       
 7   avg_surface                1030 non-null   float64       
 8   date                       1040 non-null   datetime64[ns]
 9   contract_growth_yoy        990 non-null    Float64       
 10  rent_growth_yoy            990 non-null    float64       
 11  rent_m2_growth_yoy         990 non-null    float64       
 12  contra

## Resumen hasta ahora

Lo más importante es que los conteos cuadran con la lógica del pipeline:

En districts_ipc, ipc_index_q = 960 tiene sentido por los 80 registros de 2000–2001 sin IPC usable.
avg_rent_real_2025base = 950 también cuadra: a los 80 sin IPC se suman los missing originales de avg_rent.
rent_real_growth_yoy = 910 es consistente, porque al construir YoY pierdes además el primer bloque de 4 trimestres comparables dentro de cada unidad.

En neighborhood_ipc también se ve bien:

- el IPC está completo
- las variables reales tienen prácticamente el mismo soporte que las nominales
- los YoY reales caen donde era esperable: por missing originales y por el lag de 4 trimestres

En resumen, hasta aquí ya tienes una base sólida:

- panel limpio
- merge correcto con IPC
- precios nominales y reales
- crecimientos nominales y reales
- sin duplicados

Yo daría este bloque por cerrado y seguiría con el EDA. Antes de pasar a modelos, lo siguiente más lógico sería revisar visualmente tres cosas: evolución nominal vs real, distribución de los YoY, y un resumen de missing values por variable clave.

## Seguir añadiendo:

- Euríbor a 12 meses.
Tiene sentido porque afecta el coste relativo de comprar vs alquilar y el contexto financiero general. El Banco de España publica el euríbor y sus series oficiales.
- Un indicador de mercado laboral. Aquí elegiría solo uno: o tasa de paro trimestral de Cataluña, o un indicador administrativo de paro/empleo si encuentras una serie limpia para Barcelona. Idescat publica la tasa de paro trimestral de Cataluña y la serie está disponible desde 2001, lo cual encaja razonablemente con tu panel largo.


### Qué añadiría solo como capa estructural

- Renta disponible de los hogares o renta media por zona.
- Población o composición demográfica básica.
- Alguna medida simple de vulnerabilidad o presión socioeconómica.

1) Renta disponible / renta media por zona
Esta sería mi primera elección. Open Data BCN tiene un dataset de renda disponible de les llars per càpita para Barcelona, aunque está a nivel de sección censal; eso te sirve para construir una medida estructural anual de capacidad económica del territorio y luego agregarla a barrio o distrito.

2) Población o, mejor aún, densidad de población
Si quieres una variable muy simple y útil, usaría densidad de población antes que una batería demográfica grande. Open Data BCN tiene un dataset específico de densidad de población (habitantes/ha) con desglose por barrios y distritos. Eso te aporta una medida territorial limpia de presión urbana sin complicar demasiado el modelo.

3) Vulnerabilidad socioeconómica: paro registrado
Como proxy de vulnerabilidad, elegiría paro registrado. El portal de datos de Barcelona mantiene ficheros estadísticos con información oficial de población, atur registrat, llars, etc., y además hay materiales del propio Open Data BCN que remiten a datasets de atur registrat per barris. Para tu trabajo, esta probablemente es la mejor variable de vulnerabilidad porque es interpretable y está muy conectada con capacidad de pago y fragilidad del mercado local.

**MERCADO MONETARIO. TIPOS DE INTERES MERCADO INTERBANCARIO. EURIBOR. A DOCE MESES**
Frecuencia: mensual
Inicio: ene 99

Es la mejor para tu trabajo porque:es la referencia más natural para mercado inmobiliario e hipotecas, ya viene en mensual, así no tienes que limpiar una diaria
tiene histórico largo, compatible con tu periodo, luego la puedes agregar a trimestral para mergearla con alquileres.

## Euribor

In [28]:
euribor = pd.read_csv(
    "SeriesBIE[21].csv",
    sep=",",
    encoding="latin-1",
    skiprows=8,
    header=None,
    names=["period_str", "euribor_12m"]
)



In [29]:
print(euribor.head())
print(euribor.columns)
print(euribor.shape)

  period_str euribor_12m
0      ene00       3.949
1      feb00       4.111
2      mar00       4.267
3      abr00       4.365
4      may00       4.849
Index(['period_str', 'euribor_12m'], dtype='object')
(317, 2)


### convertir period string a fecha

In [30]:
# convertir a fecha

month_map = {
    "ene": "01", "feb": "02", "mar": "03", "abr": "04",
    "may": "05", "jun": "06", "jul": "07", "ago": "08",
    "sep": "09", "oct": "10", "nov": "11", "dic": "12"
}

# Limpiar string
euribor["period_str"] = euribor["period_str"].astype(str).str.strip().str.lower()

# Quedarnos solo con filas válidas tipo ene00, feb01, etc.
euribor = euribor[
    euribor["period_str"].str.match(r"^(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)\d{2}$", na=False)
].copy()

# Convertir valor a numérico
euribor["euribor_12m"] = pd.to_numeric(euribor["euribor_12m"], errors="coerce")

# Extraer mes y año
euribor["month"] = euribor["period_str"].str[:3].map(month_map)
euribor["year_2d"] = euribor["period_str"].str[3:5].astype(int)

# Pasar a 4 dígitos
euribor["year"] = 2000 + euribor["year_2d"]

# Crear fecha mensual
euribor["date"] = pd.to_datetime(
    euribor["year"].astype(str) + "-" + euribor["month"] + "-01"
)

# Dejar limpio
euribor = euribor[["date", "euribor_12m"]].sort_values("date").reset_index(drop=True)



In [31]:
print(euribor.head())
print(euribor.tail())
print(euribor.info())

        date  euribor_12m
0 2000-01-01        3.949
1 2000-02-01        4.111
2 2000-03-01        4.267
3 2000-04-01        4.365
4 2000-05-01        4.849
          date  euribor_12m
310 2025-11-01        2.217
311 2025-12-01        2.267
312 2026-01-01        2.245
313 2026-02-01        2.221
314 2026-03-01          NaN
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315 entries, 0 to 314
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         315 non-null    datetime64[ns]
 1   euribor_12m  314 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 5.1 KB
None


In [32]:
# ver los na en euribor
print("NaN en euribor_12m:", euribor["euribor_12m"].isna().sum())   

NaN en euribor_12m: 1


In [33]:
# eliminar filas con NaN en euribor_12m
euribor = euribor.dropna(subset=["euribor_12m"]).copy()
# eliminar enero y febrero de 2026
euribor = euribor.loc[euribor["date"] < "2026-01-01"].copy()

### Euribor Trimestral

In [34]:

# Euribor mensual -> trimestral


euribor["year"] = euribor["date"].dt.year
euribor["quarter"] = euribor["date"].dt.quarter

euribor_quarterly = (
    euribor
    .groupby(["year", "quarter"], as_index=False)
    .agg(euribor_12m_q=("euribor_12m", "mean"))
)

euribor_quarterly["date"] = pd.to_datetime(
    euribor_quarterly["year"].astype(str) + "-" +
    ((euribor_quarterly["quarter"] - 1) * 3 + 1).astype(str) + "-01"
)

euribor_quarterly = euribor_quarterly[["date", "year", "quarter", "euribor_12m_q"]]



In [35]:
print(euribor_quarterly.head())
print(euribor_quarterly.tail())
print(euribor_quarterly.info())

        date  year  quarter  euribor_12m_q
0 2000-01-01  2000        1       4.109000
1 2000-04-01  2000        2       4.726333
2 2000-07-01  2000        3       5.190667
3 2000-10-01  2000        4       5.097333
4 2001-01-01  2001        1       4.545333
          date  year  quarter  euribor_12m_q
99  2024-10-01  2024        4       2.544333
100 2025-01-01  2025        1       2.443333
101 2025-04-01  2025        2       2.101667
102 2025-07-01  2025        3       2.121667
103 2025-10-01  2025        4       2.223667
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           104 non-null    datetime64[ns]
 1   year           104 non-null    int32         
 2   quarter        104 non-null    int32         
 3   euribor_12m_q  104 non-null    float64       
dtypes: datetime64[ns](1), float64(1), int32(2)
memory usage: 

### Merge con districts_ipc y neighborhood_ipc

In [40]:
# MERGE Euribor trimestral

districts_ipc_euribor = districts_ipc.merge(
    euribor_quarterly[["date", "euribor_12m_q"]],
    on="date",
    how="left"
)

neighborhood_ipc_euribor = neighborhood_ipc.merge(
    euribor_quarterly[["date", "euribor_12m_q"]],
    on="date",
    how="left"
)



In [41]:
# Revisión rápida
print(districts_ipc_euribor[["date", "avg_rent", "ipc_index_q", "euribor_12m_q"]].head())
print(neighborhood_ipc_euribor[["date", "avg_rent", "ipc_index_q", "euribor_12m_q"]].head())

print("NaN Euribor districts:", districts_ipc_euribor["euribor_12m_q"].isna().sum())
print("NaN Euribor neighborhood:", neighborhood_ipc_euribor["euribor_12m_q"].isna().sum())

        date  avg_rent  ipc_index_q  euribor_12m_q
0 2000-01-01  286.9246          NaN       4.109000
1 2000-04-01  301.3140          NaN       4.726333
2 2000-07-01  338.7222          NaN       5.190667
3 2000-10-01  370.0641          NaN       5.097333
4 2001-01-01  345.2979          NaN       4.545333
        date    avg_rent  ipc_index_q  euribor_12m_q
0 2014-01-01  589.548624    78.679000       0.562667
1 2014-04-01  550.631711    79.547000       0.569667
2 2014-07-01  576.454232    79.032333       0.439667
3 2014-10-01  596.995494    79.193333       0.334000
4 2015-01-01  601.307029    78.153667       0.255000
NaN Euribor districts: 0
NaN Euribor neighborhood: 0


## Index

Indice socioeconomico territorial por barrios

In [45]:
ist_neighborhood = pd.read_csv(
    "ist14058mun.csv",
    sep=";",
    encoding="utf-8-sig",
    decimal=","
)


In [46]:

ist_neighborhood = ist_neighborhood.rename(columns={
    "año": "year",
    "municipio": "municipality",
    "barrios de Barcelona": "neighborhood_name",
    "concepto": "concept",
    "estado": "status",
    "valor": "ist_index"
})





In [47]:

print(ist_neighborhood.head())
print(ist_neighborhood.columns)
print(ist_neighborhood.shape)
print(ist_neighborhood.info())

   year municipality                      neighborhood_name  \
0  2015    Barcelona                               el Raval   
1  2015    Barcelona                         el Barri Gòtic   
2  2015    Barcelona                         la Barceloneta   
3  2015    Barcelona  Sant Pere, Santa Caterina i la Ribera   
4  2015    Barcelona                          el Fort Pienc   

                             concept  status  ist_index  
0  Índice socioeconómico territorial     NaN       74.3  
1  Índice socioeconómico territorial     NaN       98.0  
2  Índice socioeconómico territorial     NaN       94.8  
3  Índice socioeconómico territorial     NaN       99.5  
4  Índice socioeconómico territorial     NaN      114.3  
Index(['year', 'municipality', 'neighborhood_name', 'concept', 'status',
       'ist_index'],
      dtype='object')
(666, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 666 entries, 0 to 665
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtyp

In [49]:
# 1. Dejar IST limpio

ist_neighborhood_clean = ist_neighborhood[[
    "year", "neighborhood_name", "ist_index"
]].copy()


# 2. Función de normalización

def normalize_text(text):
    if pd.isna(text):
        return text
    
    text = str(text).strip().lower()
    
    # quitar acentos
    text = "".join(
        ch for ch in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(ch)
    )
    
    # normalizar apóstrofes raros
    text = text.replace("’", "'").replace("`", "'")
    
    # quitar dobles espacios
    text = " ".join(text.split())
    
    return text


# 3. Crear columnas normalizadas

ist_neighborhood_clean["neighborhood_name_norm"] = (
    ist_neighborhood_clean["neighborhood_name"].apply(normalize_text)
)

neighborhood_names = (
    neighborhood_ipc_euribor[["neighborhood"]]
    .drop_duplicates()
    .copy()
)

neighborhood_names["neighborhood_norm"] = (
    neighborhood_names["neighborhood"].apply(normalize_text)
)




In [50]:
# 4. Comparar nombres

ist_names = set(ist_neighborhood_clean["neighborhood_name_norm"].dropna().unique())
panel_names = set(neighborhood_names["neighborhood_norm"].dropna().unique())

only_in_ist = sorted(ist_names - panel_names)
only_in_panel = sorted(panel_names - ist_names)

print("Barrios solo en IST:", len(only_in_ist))
print("Barrios solo en panel:", len(only_in_panel))

print("\nEjemplos solo en IST:")
print(only_in_ist[:20])

print("\nEjemplos solo en panel:")
print(only_in_panel[:20])

Barrios solo en IST: 6
Barrios solo en panel: 5

Ejemplos solo en IST:
['el poble sec', 'la marina del prat vermell', 'sant gervasi-galvany', 'sant gervasi-la bonanova', 'sants-badal', 'total']

Ejemplos solo en panel:
['el poble sec - aei parc montjuic', 'la marina del prat vermell - aei zona franca', 'sant gervasi - galvany', 'sant gervasi - la bonanova', 'sants - badal']


In [53]:
# see neighbohood_ipc_euribor unique values in neighborhood
print(neighborhood_ipc_euribor["neighborhood"].unique())
# count unique values in neighborhood_ipc_euribor neighborhood
print(neighborhood_ipc_euribor["neighborhood"].nunique())
# see neighborhood_names["neighborhood_norm"]
print(neighborhood_names["neighborhood_norm"].unique())
#count unique values in neighborhood_names["neighborhood_norm"]
print(neighborhood_names["neighborhood_norm"].nunique())

['el Raval' 'el Barri Gòtic' 'la Barceloneta'
 'Sant Pere, Santa Caterina i la Ribera' 'el Fort Pienc'
 'la Sagrada Família' "la Dreta de l'Eixample"
 "l'Antiga Esquerra de l'Eixample" "la Nova Esquerra de l'Eixample"
 'Sant Antoni' 'el Poble Sec - AEI Parc Montjuïc'
 'la Marina del Prat Vermell - AEI Zona Franca' 'la Marina de Port'
 'la Font de la Guatlla' 'Hostafrancs' 'la Bordeta' 'Sants - Badal'
 'Sants' 'les Corts' 'la Maternitat i Sant Ramon' 'Pedralbes'
 'Vallvidrera, el Tibidabo i les Planes' 'Sarrià' 'les Tres Torres'
 'Sant Gervasi - la Bonanova' 'Sant Gervasi - Galvany'
 'el Putxet i el Farró' 'Vallcarca i els Penitents' 'el Coll' 'la Salut'
 'la Vila de Gràcia' "el Camp d'en Grassot i Gràcia Nova"
 'el Baix Guinardó' 'Can Baró' 'el Guinardó' "la Font d'en Fargues"
 'el Carmel' 'la Teixonera' 'Sant Genís dels Agudells' 'Montbau'
 "la Vall d'Hebron" 'la Clota' 'Horta' 'Vilapicina i la Torre Llobeta'
 'Porta' 'el Turó de la Peira' 'Can Peguera' 'la Guineueta' 'Canyelles'
 'le

### Correcinn en diferencia de nombres de barrios

In [54]:
# 5. Eliminar fila "total" y mapear equivalencias manuales


ist_neighborhood_clean = ist_neighborhood_clean[
    ist_neighborhood_clean["neighborhood_name_norm"] != "total"
].copy()

manual_name_map = {
    "el poble sec": "el poble sec - aei parc montjuic",
    "la marina del prat vermell": "la marina del prat vermell - aei zona franca",
    "sant gervasi-galvany": "sant gervasi - galvany",
    "sant gervasi-la bonanova": "sant gervasi - la bonanova",
    "sants-badal": "sants - badal"
}

ist_neighborhood_clean["neighborhood_norm_matched"] = (
    ist_neighborhood_clean["neighborhood_name_norm"]
    .replace(manual_name_map)
)



In [55]:
# 6. Revisión final de mismatches

ist_names_final = set(ist_neighborhood_clean["neighborhood_norm_matched"].dropna().unique())
panel_names_final = set(neighborhood_names["neighborhood_norm"].dropna().unique())

only_in_ist_final = sorted(ist_names_final - panel_names_final)
only_in_panel_final = sorted(panel_names_final - ist_names_final)

print("Barrios solo en IST tras mapping:", len(only_in_ist_final))
print("Barrios solo en panel tras mapping:", len(only_in_panel_final))

print("\nSolo en IST:")
print(only_in_ist_final)

print("\nSolo en panel:")
print(only_in_panel_final)

Barrios solo en IST tras mapping: 0
Barrios solo en panel tras mapping: 0

Solo en IST:
[]

Solo en panel:
[]


### Merge con neighborhood_ipc_euribor

In [56]:
# MERGE IST en neighborhood panel

# Crear columna auxiliar normalizada en el panel principal
neighborhood_ipc_euribor["neighborhood_norm"] = (
    neighborhood_ipc_euribor["neighborhood"].apply(normalize_text)
)

# Hacer merge usando year + nombre normalizado
neighborhood_ipc_euribor_ist = neighborhood_ipc_euribor.merge(
    ist_neighborhood_clean[["year", "neighborhood_norm_matched", "ist_index"]],
    left_on=["year", "neighborhood_norm"],
    right_on=["year", "neighborhood_norm_matched"],
    how="left"
)

# Opcional: eliminar columna auxiliar del IST si no la quieres repetir
neighborhood_ipc_euribor_ist = neighborhood_ipc_euribor_ist.drop(
    columns=["neighborhood_norm_matched"]
)



In [57]:
# Revisión rápida
print(
    neighborhood_ipc_euribor_ist[
        ["year", "neighborhood", "ist_index"]
    ].head(10)
)

print("NaN IST:", neighborhood_ipc_euribor_ist["ist_index"].isna().sum())
print("Barrios únicos originales:", neighborhood_ipc_euribor_ist["neighborhood"].nunique())

   year neighborhood  ist_index
0  2014     el Raval        NaN
1  2014     el Raval        NaN
2  2014     el Raval        NaN
3  2014     el Raval        NaN
4  2015     el Raval       74.3
5  2015     el Raval       74.3
6  2015     el Raval       74.3
7  2015     el Raval       74.3
8  2016     el Raval       74.4
9  2016     el Raval       74.4
NaN IST: 876
Barrios únicos originales: 73


In [61]:
# comparativa entre numero de columnas de neighboorhood_ipc_euribor_ist y neighborhood_ipc_euribor
print("Columnas en neighborhood_ipc_euribor:", neighborhood_ipc_euribor.columns)
print('Número de columnas en neighborhood_ipc_euribor:', len(neighborhood_ipc_euribor.columns))
print("Columnas en neighborhood_ipc_euribor_ist:", neighborhood_ipc_euribor_ist.columns)
print('Número de columnas en neighborhood_ipc_euribor_ist:', len(neighborhood_ipc_euribor_ist.columns))

Columnas en neighborhood_ipc_euribor: Index(['neighborhood_code', 'neighborhood', 'year', 'quarter', 'num_contracts',
       'avg_rent', 'avg_rent_m2', 'avg_surface', 'date', 'contract_growth_yoy',
       'rent_growth_yoy', 'rent_m2_growth_yoy', 'contract_growth_qoq',
       'rent_m2_lag1', 'rent_m2_lag4', 'post_regulation', 'covid_dummy',
       'ipc_index_q', 'avg_rent_real_2025base', 'avg_rent_m2_real_2025base',
       'rent_real_growth_yoy', 'rent_m2_real_growth_yoy', 'euribor_12m_q',
       'neighborhood_norm'],
      dtype='object')
Número de columnas en neighborhood_ipc_euribor: 24
Columnas en neighborhood_ipc_euribor_ist: Index(['neighborhood_code', 'neighborhood', 'year', 'quarter', 'num_contracts',
       'avg_rent', 'avg_rent_m2', 'avg_surface', 'date', 'contract_growth_yoy',
       'rent_growth_yoy', 'rent_m2_growth_yoy', 'contract_growth_qoq',
       'rent_m2_lag1', 'rent_m2_lag4', 'post_regulation', 'covid_dummy',
       'ipc_index_q', 'avg_rent_real_2025base', 'avg_rent_

Como variable estructural complementaria, se incorporó al panel de barrios el Índice socioeconómico territorial (IST) publicado por Idescat. Esta variable resume la posición socioeconómica relativa de cada territorio y se utiliza como una capa adicional para capturar diferencias estructurales entre barrios más allá de la evolución coyuntural de precios y contratos.

La integración se realizó mediante un merge por año y barrio, utilizando una clave armonizada de nombres para asegurar la correspondencia entre los barrios del panel de alquiler y los barrios reportados por Idescat. Tras este proceso, el emparejamiento entre ambas fuentes fue satisfactorio y se mantuvieron intactos los 73 barrios originales del panel principal.

Los valores faltantes (NaN) observados tras el merge son esperables y no responden a errores de integración, sino a la propia cobertura temporal del IST. En concreto, esta fuente comienza en 2015, por lo que el año 2014 queda naturalmente sin información. Del mismo modo, la versión descargada no cubre todavía los años más recientes del panel, por lo que 2024 y 2025 también aparecen sin valor observado. Dado que el panel trimestral asigna el valor anual del IST a los cuatro trimestres de cada año, estos huecos temporales se reflejan en varios registros, pero su origen es puramente metodológico.

En esta etapa del trabajo, se ha optado por preservar estos valores faltantes tal como vienen de la fuente original, ya que el objetivo actual es construir un dataset base limpio y trazable para el EDA y los modelos posteriores. Más adelante, en función de las necesidades analíticas, podrá evaluarse una decisión adicional para los años finales —por ejemplo, aplicar carry-forward del último valor disponible hacia 2024 y 2025—, tratándolo siempre como una aproximación estructural lenta y no como un dato observado. Esta decisión, en cualquier caso, se reservará para la fase de modelado o análisis de robustez, y no para la construcción inicial del dataset.

## Conclusión hasta este punto

Hasta este punto, el panel ya incorpora una combinación de variables que permite enriquecer el análisis más allá de los indicadores originales del mercado de alquiler. En primer lugar, se añadieron variables macroeconómicas de contexto, como el IPC, utilizado para construir versiones reales de los precios del alquiler, y el Euríbor a 12 meses, incorporado como proxy del entorno financiero e hipotecario. En segundo lugar, se incorporó el Índice socioeconómico territorial (IST) de Idescat como variable estructural del territorio, con el objetivo de capturar, en una única medida sintética, diferencias socioeconómicas entre barrios asociadas a renta, situación laboral, nivel educativo y otras dimensiones de vulnerabilidad relativa. Con ello, el panel ya dispone de una capa macroeconómica y otra socioeconómica que complementan de forma razonable la información original sobre contratos y precios. Como siguiente paso, se añadirá la densidad de población, ya que esta variable aporta una dimensión distinta, más vinculada a la presión urbana y a la intensidad del uso residencial del espacio. Su incorporación permitirá complementar la lectura socioeconómica del IST con una medida territorial más física, útil para interpretar diferencias estructurales entre barrios y distritos.

## Población

In [63]:
url = "https://opendata-ajuntament.barcelona.cat/data/api/3/action/package_show"
params = {"id": "pad_mdbas"}

resp = requests.get(url, params=params, timeout=30)
resp.raise_for_status()

data = resp.json()

# Verifica que la llamada fue bien
print("success:", data["success"])

# Extraer recursos
resources = data["result"]["resources"]

resources_df = pd.DataFrame([
    {
        "resource_id": r.get("id"),
        "name": r.get("name"),
        "format": r.get("format"),
        "last_modified": r.get("last_modified"),
        "url": r.get("url")
    }
    for r in resources
])

print(resources_df[["name", "format", "resource_id", "last_modified"]])

success: True
                   name format                           resource_id  \
0    2025_pad_mdbas.csv    CSV  eb82adf2-a7b0-40e6-9624-b4b9eff23018   
1   2025_pad_mdbas.json   JSON  9c78eba1-7b4a-4265-bf66-96488f1e24be   
2    2024_pad_mdbas.csv    CSV  fc597601-a291-4811-ad02-c58e32784692   
3   2024_pad_mdbas.json   JSON  3cfa8831-a63b-454d-b278-042dc487ed34   
4    2023_pad_mdbas.csv    CSV  f70020ae-c41a-438d-8983-a372628c197b   
5   2023_pad_mdbas.json   JSON  6cdb1d7d-d43f-41cf-9070-0d4a91030cd3   
6    2022_pad_mdbas.csv    CSV  78b965d3-b2dc-4f23-91b9-59caa45bc334   
7   2022_pad_mdbas.json   JSON  b5a45380-ae78-4a45-bfe9-61b4281d3e16   
8    2021_pad_mdbas.csv    CSV  8b1f1cb6-14d1-4227-b03f-111bb721f82c   
9   2021_pad_mdbas.json   JSON  e43c49c5-84c8-4790-ac3d-f7879d3a7cd0   
10   2020_pad_mdbas.csv    CSV  76907f41-31fb-4afc-b5df-ae0d06d99a37   
11  2020_pad_mdbas.json   JSON  2f90dd5e-4a4f-4b2c-a704-ba999c76e094   
12   2019_pad_mdbas.csv    CSV  d1260b9e-5009-4afc

In [66]:
resources_csv = resources_df[resources_df["format"].str.upper().eq("CSV")].copy()

resources_csv["year"] = (
    resources_csv["name"]
    .str.extract(r"(\d{4})", expand=False)
    .astype("Int64")
)

population_resources = (
    resources_csv.loc[resources_csv["year"].between(2014, 2025)]
    .sort_values("year")
    .reset_index(drop=True)
)

print(population_resources[["year", "name", "resource_id", "url"]])

    year                name                           resource_id  \
0   2014  2014_pad_mdbas.csv  2edf0aab-7751-4a1d-b7b5-780079c85110   
1   2015  2015_pad_mdbas.csv  68af79ac-1df8-481a-9d03-34e5ce72e238   
2   2016  2016_pad_mdbas.csv  e32cf831-2ec5-4478-b3ea-6ea9adae5c5e   
3   2017  2017_pad_mdbas.csv  7b844962-fd60-49d0-9604-1ba4711f1450   
4   2018  2018_pad_mdbas.csv  c661f45e-a959-4193-8ff5-43a728b2bb9c   
5   2019  2019_pad_mdbas.csv  d1260b9e-5009-4afc-85f2-79baf4c0d91b   
6   2020  2020_pad_mdbas.csv  76907f41-31fb-4afc-b5df-ae0d06d99a37   
7   2021  2021_pad_mdbas.csv  8b1f1cb6-14d1-4227-b03f-111bb721f82c   
8   2022  2022_pad_mdbas.csv  78b965d3-b2dc-4f23-91b9-59caa45bc334   
9   2023  2023_pad_mdbas.csv  f70020ae-c41a-438d-8983-a372628c197b   
10  2024  2024_pad_mdbas.csv  fc597601-a291-4811-ad02-c58e32784692   
11  2025  2025_pad_mdbas.csv  eb82adf2-a7b0-40e6-9624-b4b9eff23018   

                                                  url  
0   https://opendata-ajuntament.b

In [70]:
# Inspeccionar un recurso CSV de ejemplo de pad_mdbas
sample_url = population_resources.loc[
    population_resources["year"].eq(2024), "url"
].iloc[0]

pop_sample = pd.read_csv(
    sample_url,
    sep=",",
    encoding="utf-8-sig"
)

pop_sample = pop_sample.rename(columns={
    "Data_Referencia": "date",
    "Codi_Districte": "district_code",
    "Nom_Districte": "district",
    "Codi_Barri": "neighborhood_code",
    "Nom_Barri": "neighborhood",
    "AEB": "aeb",
    "Seccio_Censal": "census_section",
    "Valor": "population"
})

print(pop_sample.head())
print(pop_sample.columns)
print(pop_sample.shape)
print(pop_sample.info())

         date  district_code      district  neighborhood_code neighborhood  \
0  2024-01-01              1  Ciutat Vella                  1     el Raval   
1  2024-01-01              1  Ciutat Vella                  1     el Raval   
2  2024-01-01              1  Ciutat Vella                  1     el Raval   
3  2024-01-01              1  Ciutat Vella                  1     el Raval   
4  2024-01-01              1  Ciutat Vella                  1     el Raval   

   aeb  census_section  population  
0    1            1001        1309  
1    1            1002        1282  
2    2            1003        3500  
3    2            1004        2999  
4    3            1005        2326  
Index(['date', 'district_code', 'district', 'neighborhood_code',
       'neighborhood', 'aeb', 'census_section', 'population'],
      dtype='object')
(1068, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1068 entries, 0 to 1067
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dty

#### Funcion para leer un recurso anual

In [78]:
def load_population_resource(url):
    df = pd.read_csv(
        url,
        sep=",",
        encoding="utf-8-sig"
    )

    df = df.rename(columns={
        "Data_Referencia": "date",
        "Codi_Districte": "district_code",
        "Nom_Districte": "district",
        "Codi_Barri": "neighborhood_code",
        "Nom_Barri": "neighborhood",
        "AEB": "aeb",
        "Seccio_Censal": "census_section",
        "Valor": "population"
    })

    df["date"] = pd.to_datetime(df["date"])
    df["year"] = df["date"].dt.year

    # forzar population a numérico
    df["population"] = pd.to_numeric(df["population"], errors="coerce")

    return df

#### Cargar todos los datos

In [79]:
population_resources_2000_2025 = (
    resources_csv.loc[resources_csv["year"].between(2000, 2025)]
    .sort_values("year")
    .reset_index(drop=True)
)

population_all = pd.concat(
    [load_population_resource(url) for url in population_resources_2000_2025["url"]],
    ignore_index=True
)


In [80]:

print(population_all.head())
print(population_all.shape)
print(population_all["year"].min(), population_all["year"].max())
print(population_all.info())

        date  district_code      district  neighborhood_code neighborhood  \
0 2000-01-01              1  Ciutat Vella                  1     el Raval   
1 2000-01-01              1  Ciutat Vella                  1     el Raval   
2 2000-01-01              1  Ciutat Vella                  1     el Raval   
3 2000-01-01              1  Ciutat Vella                  1     el Raval   
4 2000-01-01              1  Ciutat Vella                  1     el Raval   

   aeb  census_section  population  year  
0    1            1001      1151.0  2000  
1    1            1002      1170.0  2000  
2    2            1003      2295.0  2000  
3    2            1004      2155.0  2000  
4    3            1005      2049.0  2000  
(27766, 9)
2000 2025
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27766 entries, 0 to 27765
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               27766 non-null  

#### Agregacion a barrio

In [81]:
population_neighborhood = (
    population_all
    .groupby(
        ["year", "district_code", "district", "neighborhood_code", "neighborhood"],
        as_index=False
    )
    .agg(population_total=("population", "sum"))
)


In [82]:
print(population_neighborhood.head())
print(population_neighborhood.shape)
print(population_neighborhood.info())

   year  district_code      district  neighborhood_code  \
0  2000              1  Ciutat Vella                  1   
1  2000              1  Ciutat Vella                  2   
2  2000              1  Ciutat Vella                  3   
3  2000              1  Ciutat Vella                  4   
4  2000              2      Eixample                  5   

                            neighborhood  population_total  
0                               el Raval           35333.0  
1                         el Barri Gòtic           14807.0  
2                         la Barceloneta           14911.0  
3  Sant Pere, Santa Caterina i la Ribera           19473.0  
4                          el Fort Pienc           28353.0  
(1898, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1898 entries, 0 to 1897
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   year               1898 non-null   int32  
 1   district_code 

#### Agregar a distrito


In [83]:
population_district = (
    population_all
    .groupby(
        ["year", "district_code", "district"],
        as_index=False
    )
    .agg(population_total=("population", "sum"))
)



In [84]:
print(population_district.head())
print(population_district.shape)
print(population_district.info())

   year  district_code             district  population_total
0  2000              1         Ciutat Vella           84524.0
1  2000              2             Eixample          246884.0
2  2000              3       Sants-Montjuïc          166107.0
3  2000              4            Les Corts           82443.0
4  2000              5  Sarrià-Sant Gervasi          132241.0
(260, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260 entries, 0 to 259
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   year              260 non-null    int32  
 1   district_code     260 non-null    int64  
 2   district          260 non-null    object 
 3   population_total  260 non-null    float64
dtypes: float64(1), int32(1), int64(1), object(1)
memory usage: 7.2+ KB
None


#### Check del proceso

In [85]:
print("Años population_neighborhood:", population_neighborhood["year"].min(), population_neighborhood["year"].max())
print("Años population_district:", population_district["year"].min(), population_district["year"].max())

print("Barrios únicos:", population_neighborhood["neighborhood_code"].nunique())
print("Distritos únicos:", population_district["district_code"].nunique())

Años population_neighborhood: 2000 2025
Años population_district: 2000 2025
Barrios únicos: 73
Distritos únicos: 10


In [86]:
print(population_neighborhood["population_total"].describe())
print(population_district["population_total"].describe())

count     1898.000000
mean     22145.563224
std      14459.205476
min        439.000000
25%      10369.500000
50%      21253.000000
75%      30524.000000
max      59221.000000
Name: population_total, dtype: float64
count       260.000000
mean     161662.611538
std       52937.553979
min       81074.000000
25%      121699.750000
50%      160995.000000
75%      182735.000000
max      278209.000000
Name: population_total, dtype: float64


## Superficie

In [88]:
url_surface = "https://opendata-ajuntament.barcelona.cat/data/api/3/action/package_show"
params = {"id": "est-superficie"}

resp = requests.get(url_surface, params=params, timeout=30)
resp.raise_for_status()

data = resp.json()
print("success:", data["success"])

surface_resources = pd.DataFrame([
    {
        "resource_id": r.get("id"),
        "name": r.get("name"),
        "format": r.get("format"),
        "last_modified": r.get("last_modified"),
        "url": r.get("url")
    }
    for r in data["result"]["resources"]
])

print(surface_resources[["name", "format", "resource_id", "url"]])



success: True
                   name format                           resource_id  \
0   2021_superficie.csv    CSV  bb402991-6226-4b33-a901-7d23843ec9e1   
1   2020_superficie.csv    CSV  b39f1bbd-faca-4411-a222-234a09e6d1d9   
2   2019_Superficie.csv    CSV  9cd3d989-03c2-4e39-b3a7-eec0eac53e9f   
3   2018_Superficie.csv    CSV  e6c51694-7615-4e3c-a1d3-f5f954517fc4   
4   2017_Superficie.csv    CSV  d7aa700f-c2dc-4ffb-b5c2-62f494dd3c34   
5   2016_Superficie.csv    CSV  5e1a83dd-0a90-4d04-8f95-21aa99bfa8c3   
6   2015_Superficie.csv    CSV  a9fe58ab-cf8b-4fe4-a86c-06833c77d575   
7   2014_Superficie.csv    CSV  09e99391-6ecd-4ee6-9905-8ab425e883d0   
8   2013_Superficie.csv    CSV  60983406-4977-44ee-996a-52d68f2f0399   
9   2012_Superficie.csv    CSV  f14e21b2-8460-43b0-aa53-8b1115da83c1   
10  2011_Superficie.csv    CSV  55deadbc-c72f-4a2a-8cd9-08db8d37955d   
11  2010_Superficie.csv    CSV  f2a34d8f-c641-43ae-b597-158463298c44   
12  2009_Superficie.csv    CSV  fd8196b6-c3c1-40a1

In [90]:
# quedarnos con el CSV de 2021
sample_url_surface = surface_resources.loc[
    surface_resources["name"].str.contains("2021", case=False, na=False),
    "url"
].iloc[0]

surface_sample = pd.read_csv(
    sample_url_surface,
    sep=",",
    encoding="utf-8-sig"
)



In [91]:
print("URL sample:", sample_url_surface)
print(surface_sample.head())
print(surface_sample.columns)
print(surface_sample.shape)
print(surface_sample.info())

URL sample: https://opendata-ajuntament.barcelona.cat/data/dataset/8f144d2c-1185-4e5c-9b97-ac930eeffca8/resource/bb402991-6226-4b33-a901-7d23843ec9e1/download/2021_superficie.csv
    Any  Codi_Districte Nom_Districte  Codi_Barri  \
0  2021               1  Ciutat Vella           1   
1  2021               1  Ciutat Vella           2   
2  2021               1  Ciutat Vella           3   
3  2021               1  Ciutat Vella           4   
4  2021               2      Eixample           5   

                               Nom_Barri  Superfície (ha)  
0                               el Raval            110.0  
1                         el Barri Gòtic             81.6  
2                         la Barceloneta            117.9  
3  Sant Pere, Santa Caterina i la Ribera            111.0  
4                          el Fort Pienc             92.9  
Index(['Any', 'Codi_Districte', 'Nom_Districte', 'Codi_Barri', 'Nom_Barri',
       'Superfície (ha)'],
      dtype='object')
(73, 6)
<class 'p

In [92]:
# Limpiar superficie barrio


surface_neighborhood = surface_sample.rename(columns={
    "Any": "year",
    "Codi_Districte": "district_code",
    "Nom_Districte": "district",
    "Codi_Barri": "neighborhood_code",
    "Nom_Barri": "neighborhood",
    "Superfície (ha)": "surface_ha"
}).copy()

# Como la superficie la vamos a tratar como fija, nos quedamos con columnas útiles
surface_neighborhood = surface_neighborhood[
    ["district_code", "district", "neighborhood_code", "neighborhood", "surface_ha"]
].copy()



In [93]:
print(surface_neighborhood.head())
print(surface_neighborhood.shape)
print(surface_neighborhood.info())




   district_code      district  neighborhood_code  \
0              1  Ciutat Vella                  1   
1              1  Ciutat Vella                  2   
2              1  Ciutat Vella                  3   
3              1  Ciutat Vella                  4   
4              2      Eixample                  5   

                            neighborhood  surface_ha  
0                               el Raval       110.0  
1                         el Barri Gòtic        81.6  
2                         la Barceloneta       117.9  
3  Sant Pere, Santa Caterina i la Ribera       111.0  
4                          el Fort Pienc        92.9  
(73, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   district_code      73 non-null     int64  
 1   district           73 non-null     object 
 2   neighborhood_code  73 non-null     int64  
 

In [94]:

# Construir superficie distrito


surface_district = (
    surface_neighborhood
    .groupby(["district_code", "district"], as_index=False)
    .agg(surface_ha=("surface_ha", "sum"))
)

print(surface_district.head())
print(surface_district.shape)
print(surface_district.info())




   district_code             district  surface_ha
0              1         Ciutat Vella       420.5
1              2             Eixample       746.4
2              3       Sants-Montjuïc      2288.0
3              4            Les Corts       601.0
4              5  Sarrià-Sant Gervasi      1991.5
(10, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   district_code  10 non-null     int64  
 1   district       10 non-null     object 
 2   surface_ha     10 non-null     float64
dtypes: float64(1), int64(1), object(1)
memory usage: 372.0+ bytes
None


In [95]:

# Merge población + superficie


population_neighborhood_density = population_neighborhood.merge(
    surface_neighborhood[["neighborhood_code", "surface_ha"]],
    on="neighborhood_code",
    how="left"
)

population_district_density = population_district.merge(
    surface_district[["district_code", "surface_ha"]],
    on="district_code",
    how="left"
)



In [96]:
# Calcular densidad


population_neighborhood_density["population_density_ha"] = (
    population_neighborhood_density["population_total"] /
    population_neighborhood_density["surface_ha"]
)

population_district_density["population_density_ha"] = (
    population_district_density["population_total"] /
    population_district_density["surface_ha"]
)



In [97]:

# Checks


print(population_neighborhood_density.head())
print(population_neighborhood_density[["population_total", "surface_ha", "population_density_ha"]].describe())

print(population_district_density.head())
print(population_district_density[["population_total", "surface_ha", "population_density_ha"]].describe())

print("NaN density neighborhood:", population_neighborhood_density["population_density_ha"].isna().sum())
print("NaN density district:", population_district_density["population_density_ha"].isna().sum())

   year  district_code      district  neighborhood_code  \
0  2000              1  Ciutat Vella                  1   
1  2000              1  Ciutat Vella                  2   
2  2000              1  Ciutat Vella                  3   
3  2000              1  Ciutat Vella                  4   
4  2000              2      Eixample                  5   

                            neighborhood  population_total  surface_ha  \
0                               el Raval           35333.0       110.0   
1                         el Barri Gòtic           14807.0        81.6   
2                         la Barceloneta           14911.0       117.9   
3  Sant Pere, Santa Caterina i la Ribera           19473.0       111.0   
4                          el Fort Pienc           28353.0        92.9   

   population_density_ha  
0             321.209091  
1             181.458333  
2             126.471586  
3             175.432432  
4             305.199139  
       population_total   surface_ha  

# Final Merge

In [98]:

# MERGE densidad poblacional


districts_ipc_euribor_density = districts_ipc_euribor.merge(
    population_district_density[["year", "district_code", "population_total", "population_density_ha"]],
    on=["year", "district_code"],
    how="left"
)

neighborhood_ipc_euribor_ist_density = neighborhood_ipc_euribor_ist.merge(
    population_neighborhood_density[["year", "neighborhood_code", "population_total", "population_density_ha"]],
    on=["year", "neighborhood_code"],
    how="left"
)



In [99]:
# Revisión rápida
print(
    districts_ipc_euribor_density[
        ["year", "district_code", "district", "population_total", "population_density_ha"]
    ].head()
)

print(
    neighborhood_ipc_euribor_ist_density[
        ["year", "neighborhood_code", "neighborhood", "population_total", "population_density_ha"]
    ].head()
)

print("NaN density districts:", districts_ipc_euribor_density["population_density_ha"].isna().sum())
print("NaN density neighborhood:", neighborhood_ipc_euribor_ist_density["population_density_ha"].isna().sum())

   year  district_code      district  population_total  population_density_ha
0  2000              1  Ciutat Vella           84524.0             201.008323
1  2000              1  Ciutat Vella           84524.0             201.008323
2  2000              1  Ciutat Vella           84524.0             201.008323
3  2000              1  Ciutat Vella           84524.0             201.008323
4  2001              1  Ciutat Vella           89003.0             211.659929
   year  neighborhood_code neighborhood  population_total  \
0  2014                  1     el Raval           47490.0   
1  2014                  1     el Raval           47490.0   
2  2014                  1     el Raval           47490.0   
3  2014                  1     el Raval           47490.0   
4  2015                  1     el Raval           47150.0   

   population_density_ha  
0             431.727273  
1             431.727273  
2             431.727273  
3             431.727273  
4             428.636364  
NaN

In [100]:
neighborhood_ipc_euribor_ist_density.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   neighborhood_code          3504 non-null   int64         
 1   neighborhood               3504 non-null   object        
 2   year                       3504 non-null   int64         
 3   quarter                    3504 non-null   int64         
 4   num_contracts              3429 non-null   Int64         
 5   avg_rent                   3283 non-null   float64       
 6   avg_rent_m2                3285 non-null   float64       
 7   avg_surface                3400 non-null   float64       
 8   date                       3504 non-null   datetime64[ns]
 9   contract_growth_yoy        3137 non-null   Float64       
 10  rent_growth_yoy            2916 non-null   float64       
 11  rent_m2_growth_yoy         2913 non-null   float64       
 12  contra

In [101]:
districts_ipc_euribor_density.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   district_code              1040 non-null   int64         
 1   district                   1040 non-null   object        
 2   year                       1040 non-null   int64         
 3   quarter                    1040 non-null   int64         
 4   num_contracts              1030 non-null   Int64         
 5   avg_rent                   1030 non-null   float64       
 6   avg_rent_m2                1030 non-null   float64       
 7   avg_surface                1030 non-null   float64       
 8   date                       1040 non-null   datetime64[ns]
 9   contract_growth_yoy        990 non-null    Float64       
 10  rent_growth_yoy            990 non-null    float64       
 11  rent_m2_growth_yoy         990 non-null    float64       
 12  contra

In [102]:
# drop column neighborhood_norm from neighborhood_ipc_euribor_density
neighborhood_ipc_euribor_ist_density = neighborhood_ipc_euribor_ist_density.drop(
    columns=["neighborhood_norm"]
)

In [103]:
neighborhood_ipc_euribor_ist_density.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   neighborhood_code          3504 non-null   int64         
 1   neighborhood               3504 non-null   object        
 2   year                       3504 non-null   int64         
 3   quarter                    3504 non-null   int64         
 4   num_contracts              3429 non-null   Int64         
 5   avg_rent                   3283 non-null   float64       
 6   avg_rent_m2                3285 non-null   float64       
 7   avg_surface                3400 non-null   float64       
 8   date                       3504 non-null   datetime64[ns]
 9   contract_growth_yoy        3137 non-null   Float64       
 10  rent_growth_yoy            2916 non-null   float64       
 11  rent_m2_growth_yoy         2913 non-null   float64       
 12  contra

In [ ]:
# download final datasets to parquet
# districts_ipc_euribor_density.to_parquet("districts_final.parquet", index=False)
# neighborhood_ipc_euribor_ist_density.to_parquet("neighborhood_final.parquet", index=False)
